# German Declension Trainer: Articles, Pronouns & Endings in Context

**How to use this notebook:**
1. Run **Cell 1** (setup) once by pressing `Shift + Enter`.
2. Run **Cell 2** for the **B1–C1 drill**: real sentences where one article, pronoun or adjective ending is missing (type `q` to quit, `stats` to see weak spots).
3. **Sentence Mode** (full sentences + English) and the **Plural Drill** are further down; both need Cell 1 run first.

**Adding sentences to `b1c1_corpus.tsv`:** mark each word to drill as `{word|base form|type|case|gender|reason}`, e.g.
`Wegen {des|der|Artikel|Gen|m|wegen + Genitiv} Regens …`
- type: `Artikel`, `Possessiv`, `Demonstrativ`, `Pronomen`, `Reflexiv`, `Relativpronomen`, `Adjektiv`, `n-Deklination`
- case: `Nom`, `Akk`, `Dat`, `Gen` · gender: `m`, `f`, `n`, `pl`, or `-` (pronouns without gender)

In [ ]:
import csv
import random
import re
from collections import defaultdict

CASES = {"Nom": "Nominativ", "Akk": "Akkusativ", "Dat": "Dativ", "Gen": "Genitiv"}
WORD_TYPES = ("Artikel", "Possessiv", "Demonstrativ", "Pronomen", "Reflexiv", "Relativpronomen",
              "Adjektiv", "n-Deklination")
MARKER = re.compile(r"\{([^{}|]+)\|([^{}|]+)\|([^{}|]+)\|([^{}|]+)\|([^{}|]+)\|([^{}]+)\}")

def load_b1c1(path="b1c1_corpus.tsv"):
    """One entry per marked word: the sentence with that word blanked, the others filled in."""
    items = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            text = row["german"]
            marks = list(MARKER.finditer(text))
            assert marks, f"No {{…}} markers in {path}: {text}"
            plain = MARKER.sub(lambda m: m.group(1), text)
            for m in marks:
                word, base, wtype, case, gn, reason = (g.strip() for g in m.groups())
                where = f"{path}: {plain}"
                assert wtype in WORD_TYPES, f"Unknown type {wtype!r} ({where})"
                assert case in CASES, f"Unknown case {case!r} ({where})"
                assert gn in ("m", "f", "n", "pl", "-"), f"Unknown gender {gn!r} ({where})"
                gap = (MARKER.sub(lambda o: o.group(1), text[:m.start()]) + f"___ ({base})"
                       + MARKER.sub(lambda o: o.group(1), text[m.end():]))
                items.append({"level": row["level"], "topic": row["topic"], "english": row["english"],
                              "sentence": plain, "gap": gap, "answer": word, "base": base,
                              "type": wtype, "case": CASES[case], "gn": gn, "reason": reason})
    return items

B1C1_ITEMS = load_b1c1()

def check_word(user_ans, answer):
    """Return 'exact', 'case' (right except capitalisation) or None."""
    ans = normalize(user_ans)
    if ans == answer:
        return "exact"
    if ans.casefold() == answer.casefold():
        return "case"
    return None

def describe(item):
    """e.g. 'Artikel · Genitiv · maskulin · Singular'"""
    parts = [item["type"], item["case"]]
    if item["gn"] == "pl":
        parts.append("Plural")
    elif item["gn"] != "-":
        parts += [SLOT_LABELS[item["gn"]], "Singular"]
    return " · ".join(parts)

def normalize(text):
    """Collapse extra/leading/trailing whitespace."""
    return " ".join(text.split())

def check_answer(user_ans, answers):
    """Return 'exact', 'case' (right except capitalisation) or None."""
    ans = normalize(user_ans)
    # The article may be capitalised (sentence start); only the noun's capitalisation matters.
    art, _, noun = ans.partition(" ")
    if any(art.casefold() == a_art.casefold() and noun == a_noun
           for a_art, _, a_noun in (a.partition(" ") for a in answers)):
        return "exact"
    if ans.casefold() in (a.casefold() for a in answers):
        return "case"
    return None

SLOT_LABELS = {"m": "maskulin", "f": "feminin", "n": "neutral", "pl": "Plural", "-": "ohne Genus"}

def new_stats():
    # key -> [correct, total]
    return {"case": defaultdict(lambda: [0, 0]), "slot": defaultdict(lambda: [0, 0])}

def record(stats, case, slot, correct):
    for key, value in (("case", case), ("slot", slot)):
        stats[key][value][1] += 1
        stats[key][value][0] += int(correct)

def print_stats(stats):
    if not stats["case"]:
        print("No answers yet.")
        return
    for key, title in (("case", "By case"), ("slot", "By gender / number"), ("type", "By word type"),
                       ("level", "By level")):
        if key not in stats:
            continue
        print(f"\n{title} (weakest first):")
        rows = sorted(stats[key].items(), key=lambda kv: kv[1][0] / kv[1][1])
        for value, (right, n) in rows:
            label = SLOT_LABELS.get(value, value)
            print(f"  {label:<16} {right:>3}/{n:<3} ({round(right / n * 100)}%)")

print(f"Setup complete! {len(B1C1_ITEMS)} gaps loaded from b1c1_corpus.tsv. Run the next cell to start drilling.")

In [ ]:
# B1–C1 Drill: fill in the missing article / pronoun / adjective ending
FOCUS_LEVEL = None   # "B1", "B2", "C1", or None for all
FOCUS_CASE = None    # "Nominativ", "Akkusativ", "Dativ", "Genitiv", or None
FOCUS_TYPE = None    # e.g. "Adjektiv", "Relativpronomen", "Possessiv", or None

pool = [it for it in B1C1_ITEMS
        if FOCUS_LEVEL in (None, it["level"]) and FOCUS_CASE in (None, it["case"])
        and FOCUS_TYPE in (None, it["type"])]
random.shuffle(pool)

score = total = 0
mistakes = []
stats = new_stats()
stats["type"] = defaultdict(lambda: [0, 0])
stats["level"] = defaultdict(lambda: [0, 0])

print(f"{len(pool)} gaps. Type 'q' to finish, 'stats' to see your weak spots.\n" + "-" * 55)

for it in pool:
    print(f"\n[{it['level']} · {it['topic']}]  {it['gap']}")
    print(f"   ({it['english']})")
    while True:
        user_ans = input("Your answer: ").strip()
        if user_ans.lower() == "stats":
            print_stats(stats)
            continue
        break
    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    result = check_word(user_ans, it["answer"])
    record(stats, it["case"], it["gn"], result is not None)
    for key in ("type", "level"):
        stats[key][it[key]][1] += 1
        stats[key][it[key]][0] += int(result is not None)

    if result is not None:
        score += 1
        print(f"✅ Richtig! ({score}/{total})  {it['answer']} → {describe(it)}")
        if result == "case" and it["type"] == "n-Deklination":
            print(f"   ⚠️ Nomen schreibt man groß: '{it['answer']}'")
    else:
        mistakes.append((it, user_ans))
        print(f"❌ Falsch! Dein Input: '{user_ans}'  →  Richtig: '{it['answer']}'  ({describe(it)})")
    print(f"   Grund: {it['reason']}")
    print(f"   → {it['sentence']}")
else:
    print("\n🎉 You went through all the gaps!")

print("\n" + "=" * 55)
if total:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_stats(stats)
    if mistakes:
        print("\nReview these slips:")
        for it, u in mistakes:
            print(f"  • {it['gap']}\n    You wrote: {u}  |  Correct: {it['answer']}  ({it['reason']})")
else:
    print("Session ended.")

## Sentence Mode: Real sentences + case triggers

Loads `sentence_corpus.tsv` (180 sentences with English translations). Fill the gap with **article + noun**.
Each prompt shows the case trigger (verb, preposition, wo?/wohin?, Genitivattribut, …) and the noun in its dictionary form.
Edit the TSV in any spreadsheet app to add your own sentences. Separate alternative answers with `|`.

In [ ]:
# Sentence Mode Drill (needs Cell 1 to be run first)
import csv

FOCUS_CASE = None          # e.g. "Dativ", or None for all cases
FOCUS_TRIGGER_TYPE = None  # e.g. "Wechselpräposition" (substring match), or None
SHOW_ENGLISH_FIRST = False # True = show the translation before you answer

with open("sentence_corpus.tsv", encoding="utf-8") as f:
    CORPUS = list(csv.DictReader(f, delimiter="\t"))

pool = [r for r in CORPUS
        if (FOCUS_CASE is None or r["case"] == FOCUS_CASE)
        and (FOCUS_TRIGGER_TYPE is None or FOCUS_TRIGGER_TYPE.lower() in r["trigger_type"].lower())]
print(f"{len(pool)} sentences loaded. Type 'q' to quit, 'stats' for weak spots.\n" + "-" * 55)

GENDER_OF = {"der": "m", "die": "f", "das": "n"}
score = total = 0
mistakes = []
stats = new_stats()
random.shuffle(pool)

for row in pool:
    answers = [a.strip() for a in row["answer"].split("|")]
    slot = "pl" if row["number"] == "pl" else GENDER_OF[row["noun"].split()[0]]
    num = " (Pl.)" if row["number"] == "pl" else ""
    print(f"\n[{row['trigger']} · {row['trigger_type']}]   ({row['noun']}{num}, {row['article']})")
    print(f"  {row['german']}")
    if SHOW_ENGLISH_FIRST:
        print(f"  {row['english']}")

    while True:
        user_ans = input("Your answer: ").strip()
        if user_ans.lower() == "stats":
            print_stats(stats)
            continue
        break
    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    result = check_answer(user_ans, answers)
    record(stats, row["case"], slot, result is not None)
    full = row["german"].replace("___", answers[0])
    if result is not None:
        score += 1
        print(f"✅ Richtig! ({score}/{total})  →  {full}")
        if result == "case":
            print(f"   ⚠️ Achte auf die Großschreibung: '{answers[0]}'")
        if len(answers) > 1:
            print(f"   (Beide Formen sind korrekt: {' / '.join(answers)})")
    else:
        mistakes.append((row["german"], user_ans, " / ".join(answers), row["case"], row["trigger"]))
        print(f"❌ Falsch! ({row['case']})  →  {full}")
    if not SHOW_ENGLISH_FIRST:
        print(f"   {row['english']}")
else:
    print("\n🎉 You went through the whole corpus!")

print("\n" + "=" * 55)
if total:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_stats(stats)
    if mistakes:
        print("\nReview these slips:")
        for s, u, c, case, trig in mistakes:
            print(f"  • {s}  [{case} ← {trig}]\n    You wrote: {u}  |  Correct: {c}")
else:
    print("Session ended.")

## Plural Drill: Learn the plural patterns

Loads `plural_nouns.tsv` (149 nouns, each tagged with its plural pattern). You see `der Stuhl (chair)` → type the plural (`Stühle`, with or without `die`).
On a mistake it tells you **which rule applies** and spots a missing or wrong umlaut. Type `ae/oe/ue` if you have no umlaut keys.
Add your own nouns to the TSV; the `pattern` column must be one of the keys in `PLURAL_RULES` below.

In [ ]:
# Plural Drill
import csv

FOCUS_GENDER = None    # "der", "die", "das", or None for all
FOCUS_PATTERN = None   # a key from PLURAL_RULES (e.g. "m_uml_e"), or None

PLURAL_RULES = {
    "f_en":       "Feminin → -(e)n. -ung, -heit, -keit, -schaft, -ion, -tät, -ik, -ei always take -en (~90% of feminine nouns).",
    "f_n":        "Feminin on -e, -el, -er → just add -n.",
    "f_nen":      "Feminin on -in → -innen (double n!).",
    "f_uml_e":    "Exception: short feminine nouns (Stadt, Hand, Nacht, Wand, Wurst, Maus) → ¨-e.",
    "f_uml":      "Exception: die Mutter, die Tochter → umlaut only, no ending.",
    "nis":        "-nis → -nisse (the s doubles).",
    "m_weak":     "n-Deklination (masc. -e people/animals, -ent, -ant, -ist, -oge, + Mensch, Nachbar, Herr, Name, Gedanke) → -(e)n.",
    "m_e":        "Masculine → -e, here WITHOUT umlaut. Memorise it: der Tag, -e.",
    "m_uml_e":    "Masculine → -e WITH umlaut. Memorise it: der Stuhl, ¨-e.",
    "m_zero":     "Masculine on -el, -en, -er → no ending, no umlaut.",
    "m_uml_zero": "Masculine on -el, -en, -er → no ending, but WITH umlaut.",
    "m_uml_er":   "Small masculine group → ¨-er (Mann, Wald, Mund, Gott).",
    "n_er":       "Short neuter nouns → -er, with umlaut wherever possible (a/o/u/au).",
    "n_e":        "Neuter → -e, and neuter -e plurals NEVER take an umlaut.",
    "n_zero":     "Neuter on -chen, -lein, -el, -er, -en or Ge-…-e → no change.",
    "n_en":       "Small neuter group → -(e)n (Auge, Ende, Ohr, Bett, Hemd, Herz).",
    "um":         "-um → -en (Museen, Zentren) or -a (Praktika).",
    "s":          "Foreign words and nouns ending in a vowel (-o, -a, -y, -é) → -s.",
}

with open("plural_nouns.tsv", encoding="utf-8") as f:
    NOUNS = list(csv.DictReader(f, delimiter="\t"))
unknown = {r["pattern"] for r in NOUNS} - PLURAL_RULES.keys()
assert not unknown, f"Unknown pattern(s) in plural_nouns.tsv: {unknown}"

UMLAUT_STRIP = str.maketrans("äöüÄÖÜ", "aouAOU")
def translit(s):
    for a, b in (("ä", "ae"), ("ö", "oe"), ("ü", "ue"), ("Ä", "Ae"), ("Ö", "Oe"), ("Ü", "Ue")):
        s = s.replace(a, b)
    return s

def check_plural(user, correct):
    """Return (ok, note)."""
    ans = " ".join(user.split())
    if ans.lower().startswith("die "):
        ans = ans[4:]
    if ans == correct:
        return True, None
    if ans == translit(correct):
        return True, f"Mit Umlaut geschrieben: {correct}"
    if ans.casefold() in (correct.casefold(), translit(correct).casefold()):
        return True, f"Achte auf die Großschreibung: {correct}"
    if ans.translate(UMLAUT_STRIP) == correct.translate(UMLAUT_STRIP) or translit(ans) == translit(correct):
        if correct.translate(UMLAUT_STRIP) == correct:
            return False, "No umlaut here!"
        return False, "Umlaut missing!" if ans.translate(UMLAUT_STRIP) == ans else "Umlaut on the wrong vowel!"
    return False, None

pool = [r for r in NOUNS
        if (FOCUS_GENDER is None or r["article"] == FOCUS_GENDER)
        and (FOCUS_PATTERN is None or r["pattern"] == FOCUS_PATTERN)]
random.shuffle(pool)
print(f"{len(pool)} nouns loaded. Type 'q' to quit, 'stats' for weak spots.\n" + "-" * 55)

pstats = {"pattern": defaultdict(lambda: [0, 0]), "article": defaultdict(lambda: [0, 0])}
def print_plural_stats():
    if not pstats["pattern"]:
        print("No answers yet.")
        return
    for key, title in (("article", "By gender"), ("pattern", "By plural pattern")):
        print(f"\n{title} (weakest first):")
        for value, (right, n) in sorted(pstats[key].items(), key=lambda kv: kv[1][0] / kv[1][1]):
            print(f"  {value:<11} {right:>3}/{n:<3} ({round(right / n * 100)}%)")

score = total = 0
mistakes = []
for row in pool:
    print(f"\n{row['article']} {row['singular']} ({row['english']})  →  die ___")
    while True:
        user_ans = input("Plural: ").strip()
        if user_ans.lower() == "stats":
            print_plural_stats()
            continue
        break
    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    ok, note = check_plural(user_ans, row["plural"])
    for key in ("pattern", "article"):
        pstats[key][row[key]][1] += 1
        pstats[key][row[key]][0] += int(ok)
    if ok:
        score += 1
        print(f"✅ Richtig! die {row['plural']} ({score}/{total})")
        if note:
            print(f"   ⚠️ {note}")
    else:
        mistakes.append((row, user_ans))
        print(f"❌ Falsch!{' ' + note if note else ''}  →  die {row['plural']}")
        print(f"   Regel: {PLURAL_RULES[row['pattern']]}")
else:
    print("\n🎉 You went through all the nouns!")

print("\n" + "=" * 55)
if total:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_plural_stats()
    if mistakes:
        print("\nReview these slips:")
        for row, u in mistakes:
            print(f"  • {row['article']} {row['singular']} → die {row['plural']}   (you wrote: {u})")
else:
    print("Session ended.")